# Regression cutpoint tuning

Experiment: replace fixed **`np.rint` + clip [0, 5]** with **five learned cutpoints** for [`catboost_regressor`](../../src/modeling/models/ordinal.py) on **base daily features only** (no history).

Hyperparameters are **fixed** from the base `catboost_regressor` run in [`4 fatigue regression.ipynb`](4%20fatigue%20regression.ipynb) so this notebook isolates the effect of cutpoint calibration.

## How cutpoint tuning works

1. **Problem:** CatBoost regressor outputs a continuous fatigue score. Production regression models map that to integers 0–5 via `np.rint` then clip — equivalent to fixed cutpoints `[0.5, 1.5, 2.5, 3.5, 4.5]`.
2. **Idea:** Learn five monotonic cutpoints \(t_1 < t_2 < t_3 < t_4 < t_5\) on **out-of-fold (OOF)** continuous predictions to minimize MAE on train/val.
3. **Protocol:** Same stratified participant split as notebook 4. GroupKFold on train/val participants collects OOF continuous preds (no test leakage). Cutpoints are tuned on the stacked OOF set only.
4. **Held-out test:** Refit CatBoost on full train/val with fixed hyperparameters; apply **baseline rint** vs **tuned cutpoints** once on test for reporting.
5. **Not tuned on test:** CatBoost hyperparameters and cutpoints both come from train/val only.

```mermaid
flowchart TD
  split["Train/val + held-out test split"]
  tuneModel["Fix catboost_regressor hyperparams from nb4 base run"]
  oof["GroupKFold OOF continuous preds on train/val"]
  tuneCut["Grid-search 5 monotonic cutpoints on OOF only"]
  compareCV["Compare OOF MAE: rint vs tuned cutpoints"]
  refit["Refit CatBoost on full train/val"]
  testEval["Test MAE: rint vs tuned cutpoints"]
  split --> tuneModel --> oof --> tuneCut --> compareCV
  tuneCut --> refit --> testEval
```

Optional follow-up: if OOF or test MAE improves by **≥ 0.02**, promote cutpoints to `config.py` or wire into `OrdinalRegressorWrapper`. Optuna over cutpoints (50 trials) is another upgrade path.

In [3]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

from modeling.calibration import collect_oof_continuous_predictions, tune_cutpoints_grid
from modeling.config import DATA_PATH, N_CV_FOLDS, TIME_COL, TIME_SERIES_GROUP_COLS
from modeling.data import (
    build_split_bundle,
    load_fatigue_data,
    participant_strata,
    preprocess_after_split,
    split_participant_ids,
    split_summary_table,
)
from modeling.metrics import (
    DEFAULT_REGRESSION_CUTPOINTS,
    clip_ordinal_predictions,
    compute_metrics,
    discretize_with_cutpoints,
    mae_from_continuous,
)
from modeling.registry import ORDINAL_MODELS, make_model_factory

## 1. Load data and split

Same stratified participant hold-out as notebook 4.

In [5]:
df = load_fatigue_data('../../' + DATA_PATH)
df = df.sort_values(TIME_SERIES_GROUP_COLS + [TIME_COL]).reset_index(drop=True)

strata = participant_strata(df)
train_val_ids, test_ids = split_participant_ids(df['id'].unique(), strata=strata)
train_val_mask = df['id'].isin(train_val_ids)
test_mask = df['id'].isin(test_ids)

df = preprocess_after_split(df, train_val_mask)
bundle = build_split_bundle(df, train_val_ids, test_ids, train_val_mask, test_mask)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))

Rows: 3,178  Participants: 40


,split,participants,rows,mean_fatigue
0,train_val,32,2536,2.673107
1,test,8,642,2.151090


Test participant ids: [np.int64(8), np.int64(15), np.int64(24), np.int64(33), np.int64(40), np.int64(44), np.int64(45), np.int64(48)]


## 2. Fixed CatBoost regressor (base)

Pin hyperparameters from notebook 4 base `catboost_regressor`. Update `CATBOOST_BASE_PARAMS` after re-running nb4 with `print_tune_summary` if needed (learning_rate / l2_leaf_reg may differ from defaults below).

In [6]:
CATBOOST_BASE_PARAMS = {
    'iterations': 440,
    'depth': 4,
    # Update from nb4 print_tune_summary('catboost_regressor', ...) if re-tuned:
    'learning_rate': 0.05,
    'l2_leaf_reg': 3.0,
}

factory = make_model_factory('catboost_regressor', CATBOOST_BASE_PARAMS, ORDINAL_MODELS)
print('Model: catboost_regressor (base features, tree matrix)')
print('Fixed params:', CATBOOST_BASE_PARAMS)

Model: catboost_regressor (base features, tree matrix)
Fixed params: {'iterations': 440, 'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 3.0}


## 3. OOF continuous predictions

GroupKFold on train/val participants; each validation fold gets continuous preds from a model fit on the other folds.

In [7]:
oof_y, oof_cont = collect_oof_continuous_predictions(
    factory,
    bundle.X_train_val_tree,
    bundle.y_ord_train_val,
    bundle.groups_train_val,
    n_splits=N_CV_FOLDS,
    test_ids=bundle.test_ids,
)

print(f'OOF rows: {len(oof_y):,}')
print(f'Continuous pred range: [{oof_cont.min():.3f}, {oof_cont.max():.3f}]')

OOF rows: 2,536
Continuous pred range: [0.693, 4.340]


## 4. Tune cutpoints on OOF

Coordinate grid search (±0.4 around each cutpoint, step 0.1, 3 passes) with monotonicity constraint.

In [8]:
tune_result = tune_cutpoints_grid(oof_y, oof_cont)
best_cutpoints = tune_result['best_cutpoints']

print('Default cutpoints (rint equivalent):', DEFAULT_REGRESSION_CUTPOINTS.tolist())
print('Tuned cutpoints:', best_cutpoints.tolist())
print(f"Baseline OOF MAE (rint): {tune_result['baseline_mae']:.4f}")
print(f"Tuned OOF MAE:          {tune_result['best_mae']:.4f}")
print(f"OOF delta (tuned - rint): {tune_result['best_mae'] - tune_result['baseline_mae']:+.4f}")

display(tune_result['search_log'].tail(15))

Default cutpoints (rint equivalent): [0.5, 1.5, 2.5, 3.5, 4.5]
Tuned cutpoints: [0.5, 0.6000000000000001, 0.8, 4.4, 4.5]
Baseline OOF MAE (rint): 1.1924
Tuned OOF MAE:          1.0655
OOF delta (tuned - rint): -0.1270


,pass,index,offset,cutpoints,mae,delta_vs_best
82,3,2,-0.0,"[0.5, 0.6000000000000001, 0.8, 4.4, 4.5]",1.065457,0.000000
83,3,2,0.1,"[0.5, 0.6000000000000001, 0.9, 4.4, 4.5]",1.065852,0.000394
84,3,2,0.2,"[0.5, 0.6000000000000001, 1.0, 4.4, 4.5]",1.065852,0.000394
85,3,2,0.3,"[0.5, 0.6000000000000001, 1.1, 4.4, 4.5]",1.066640,0.001183
86,3,2,0.4,"[0.5, 0.6000000000000001, 1.2000000000000002, ...",1.067429,0.001972
87,3,3,-0.4,"[0.5, 0.6000000000000001, 0.8, 4.0, 4.5]",1.067429,0.001972
88,3,3,-0.3,"[0.5, 0.6000000000000001, 0.8, 4.1000000000000...",1.065852,0.000394
89,3,3,-0.2,"[0.5, 0.6000000000000001, 0.8, 4.2, 4.5]",1.066640,0.001183
90,3,3,-0.1,"[0.5, 0.6000000000000001, 0.8, 4.3000000000000...",1.066246,0.000789
91,3,3,-0.0,"[0.5, 0.6000000000000001, 0.8, 4.4, 4.5]",1.065457,0.000000


## 5. OOF comparison table

In [9]:
oof_pred_rint = clip_ordinal_predictions(oof_cont)
oof_pred_tuned = discretize_with_cutpoints(oof_cont, best_cutpoints)

oof_compare = pd.DataFrame(
    [
        compute_metrics(oof_y, oof_pred_rint),
        compute_metrics(oof_y, oof_pred_tuned),
    ],
    index=['rint_clip', 'tuned_cutpoints'],
)
display(oof_compare)

,mae,rmse,r2,qwk
rint_clip,1.192429,1.509434,-0.201246,0.020949
tuned_cutpoints,1.065457,1.415607,-0.056548,-0.000129


## 6. Held-out test evaluation

Single refit on all train/val; cutpoints fixed from OOF tuning above.

In [10]:
model = factory()
model.fit(bundle.X_train_val_tree, bundle.y_ord_train_val)
test_cont = model.predict_continuous(bundle.X_test_tree)
y_test = bundle.y_ord_test

test_pred_rint = clip_ordinal_predictions(test_cont)
test_pred_tuned = discretize_with_cutpoints(test_cont, best_cutpoints)

test_compare = pd.DataFrame(
    [
        compute_metrics(y_test, test_pred_rint),
        compute_metrics(y_test, test_pred_tuned),
    ],
    index=['rint_clip', 'tuned_cutpoints'],
)
display(test_compare)

delta_test_mae = test_compare.loc['tuned_cutpoints', 'mae'] - test_compare.loc['rint_clip', 'mae']
print(f'Test MAE delta (tuned - rint): {delta_test_mae:+.4f}')

,mae,rmse,r2,qwk
rint_clip,1.451713,1.780826,-0.175286,0.060930
tuned_cutpoints,1.504673,1.848633,-0.266492,0.008357


Test MAE delta (tuned - rint): +0.0530


## 7. Next steps

- If **OOF or test MAE improves by ≥ 0.02**, consider adding `REGRESSION_CUTPOINTS` to [`config.py`](../../src/modeling/config.py) or applying tuned cutpoints inside `OrdinalRegressorWrapper.predict`.
- Re-run this notebook after updating `CATBOOST_BASE_PARAMS` from notebook 4.
- Optional: replace the coordinate grid with an Optuna study over five ordered cutpoints (~50 trials).